# Notebook 00: From Text to Knowledge Graph
## Building the Foundation of a GraphRAG System using Local LLMs

Traditional Retrieval-Augmented Generation (RAG) is excellent at finding specific facts — like a needle in a haystack. However, standard vector RAG struggles with *global sensemaking* questions that require an understanding of an entire document collection (e.g., *"What are the main themes in this dataset?"*).

**GraphRAG** solves this by converting flat text into a structured Knowledge Graph. In this notebook we use a local LLM via Ollama to process a raw PDF, extract named entities and relationships, and build a network graph.

**Paper reference:** Edge et al. (2025) — *From Local to Global: A GraphRAG Approach to Query-Focused Summarization* (arXiv:2404.16130v2)

---
### What this notebook produces

```
data/input/wc2022_trimmed.pdf   (2022 FIFA World Cup, Wikipedia excerpt)
           │
           ▼  § 3.1.1  chunk at 600 tok / 100 tok overlap
        Text Chunks
           │
           ▼  § 3.1.2 + App. A  LLM extraction + glean pass
     Entities + Relationships
           │
           ▼  § 3.1.3  merge duplicates, build graph
     data/output/wc2022_trimmed_graph.json
     data/output/wc2022_trimmed_graph.html  (interactive visualisation)
```

## Imports and Setup

In [1]:
import re
import json
import subprocess
from pathlib import Path
from collections import defaultdict

import fitz                              # pymupdf
import networkx as nx
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter

Path("../data/output").mkdir(parents=True, exist_ok=True)
print("Imports OK")

Imports OK


### LLM Setup

We use **Ollama** to serve a local model. The helper below finds the server automatically so the notebook is portable: it honours an `OLLAMA_BASE_URL` environment variable if you set one, resolves the Windows-host IP when running under WSL2, and otherwise falls back to `http://localhost:11434`. The model is `qwen2.5:7b` — a good balance between quality and speed for entity extraction.

> **Why `temperature=0.0`?** Entity extraction is not a creative task. We want the model to reliably follow the tuple format from the paper, not hallucinate. Zero temperature gives deterministic, format-compliant output.

In [2]:
import os
import platform


def ollama_base_url() -> str:
    """Locate the Ollama server so this notebook runs on any machine.

    Priority: OLLAMA_BASE_URL env var  →  WSL2 Windows-host IP  →  localhost.
    Set OLLAMA_BASE_URL yourself if Ollama runs somewhere else (e.g. a remote box).
    """
    if os.environ.get("OLLAMA_BASE_URL"):
        return os.environ["OLLAMA_BASE_URL"]
    if "microsoft" in platform.uname().release.lower():   # WSL2 → Ollama on the Windows host
        host = subprocess.check_output(
            "ip route list default | awk '{print $3}'", shell=True
        ).decode().strip()
        if host:
            return f"http://{host}:11434"
    return "http://localhost:11434"                        # native Linux / macOS / Windows

MODEL    = "qwen2.5:7b"
BASE_URL = ollama_base_url()

llm = ChatOllama(
    model=MODEL,
    base_url=BASE_URL,
    temperature=0.0,
    num_predict=1024,
)

print(f"LLM: {MODEL}  @  {BASE_URL}")

LLM: qwen2.5:7b  @  http://172.19.64.1:11434


/tmp/ipykernel_51058/3417568848.py:24: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(


---
## Step 1 — Load and Chunk the PDF  *(§ 3.1.1)*

The paper chunks text at **600 tokens with 100-token overlap** (§ 3.1.1). This window is small enough that the LLM can focus on a coherent passage, while the overlap prevents relationships straddling chunk boundaries from being lost.

We use a *character-based* splitter with 1 token ≈ 4 characters → 2 400 chars / 400 chars overlap. Before chunking we strip citation markers like `[2]` or `[A]` that appear densely in Wikipedia text — they add noise without contributing to entity extraction.

In [3]:
PDF_PATH = Path("../data/input/wc2022_trimmed.pdf")

# Output paths derived from the input filename — change PDF_PATH and everything follows
GRAPH_PATH = Path("../data/output") / (PDF_PATH.stem + "_graph.json")
HTML_PATH  = Path("../data/output") / (PDF_PATH.stem + "_graph.html")

print(f"Input : {PDF_PATH}")
print(f"Graph : {GRAPH_PATH}")
print(f"HTML  : {HTML_PATH}")

# 1a: Extract text from PDF
doc = fitz.open(PDF_PATH)
raw_text = "\n".join(page.get_text() for page in doc)
print(f"\nExtracted {len(raw_text):,} characters from {len(doc)} pages")

# 1b: Clean citation markers ([2], [A], [12], etc.)
clean_text = re.sub(r"\[[\w,\s]+\]", "", raw_text)
clean_text = re.sub(r"\s{2,}", " ", clean_text)
print(f"After cleaning: {len(clean_text):,} characters")

# 1c: Chunk
CHARS_PER_TOKEN = 4
CHUNK_SIZE_TOK  = 600
OVERLAP_TOK     = 100

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE_TOK * CHARS_PER_TOKEN,
    chunk_overlap=OVERLAP_TOK * CHARS_PER_TOKEN,
)
chunks = splitter.split_text(clean_text)
print(f"\nProduced {len(chunks)} chunks  (target: ~600 tok each)")

print("\n--- Chunk 0 preview ---")
print(chunks[0][:400], "...")

Input : ../data/input/wc2022_trimmed.pdf
Graph : ../data/output/wc2022_trimmed_graph.json
HTML  : ../data/output/wc2022_trimmed_graph.html

Extracted 55,835 characters from 15 pages
After cleaning: 54,149 characters

Produced 27 chunks  (target: ~600 tok each)

--- Chunk 0 preview ---
← 2018
2026 →
2022 FIFA World Cup
2022 القدم لكرة العالم كأس
Tournament details
Host country
Qatar
Dates
20 November – 18
December
Teams
32 (from 5 confederations)
Venue
8 (in 5 host cities)
Final positions
Champions Argentina (3rd title)
Runners-up France
Third place Croatia
Fourth place Morocco
Tournament statistics
Matches played
64
Goals scored
172 (2.69 per match)
Attendance
3,404,252 (53,191 ...


---
## Step 2 — Entity & Relationship Extraction  *(§ 3.1.2, Appendix E.1)*

The core idea: ask the LLM to output every entity and relationship as **flat tuples** that we can parse deterministically. The paper defines this format in Appendix E.1.

### Delimiters (from the paper)
| Symbol | Role |
|---|---|
| `<\|>` | `tuple_delimiter` — separates fields within one tuple |
| `##` | `record_delimiter` — separates tuples from each other |
| `<\|COMPLETE\|>` | `completion_delimiter` — marks end of all output |

### Tuple formats
- **Entity**: `("entity"<|>NAME<|>TYPE<|>description)`
- **Relationship**: `("relationship"<|>source<|>target<|>description<|>strength)`

### Entity types for this dataset
`TEAM`, `NATION`, `STADIUM`, `CITY`, `CONFEDERATION`, `COMPETITION`, `PERSON`, `EVENT`

The prompt template below (shown here so you can read it alongside the paper):

In [4]:
TUPLE_DELIMITER      = "<|>"
RECORD_DELIMITER     = "##"
COMPLETION_DELIMITER = "<|COMPLETE|>"

ENTITY_TYPES = "TEAM, NATION, STADIUM, CITY, CONFEDERATION, COMPETITION, PERSON, EVENT"

EXTRACTION_PROMPT = """\
---Goal---
Given a text document, identify all entities of the listed types and all relationships
among the identified entities.

---Steps---
1. For each entity, extract:
   - entity_name: name of the entity (capitalized)
   - entity_type: one of the listed types
   - entity_description: comprehensive description of the entity's attributes and activities
   Format: ("entity"{tuple_delimiter}<entity_name>{tuple_delimiter}<entity_type>{tuple_delimiter}<entity_description>)

2. For each pair of clearly related entities, extract:
   - source_entity: name of source entity (from step 1)
   - target_entity: name of target entity (from step 1)
   - relationship_description: explanation of why source and target are related
   - relationship_strength: integer score 1-10 indicating strength
   Format: ("relationship"{tuple_delimiter}<source_entity>{tuple_delimiter}<target_entity>{tuple_delimiter}<relationship_description>{tuple_delimiter}<relationship_strength>)

3. Output all entities then all relationships as one list, using {record_delimiter} as separator.
4. End with {completion_delimiter}

---Examples---
("entity"{tuple_delimiter}FRANCE{tuple_delimiter}NATION{tuple_delimiter}France is a country competing in the 2026 FIFA World Cup)
##
("entity"{tuple_delimiter}KYLIAN MBAPPE{tuple_delimiter}PERSON{tuple_delimiter}French professional footballer and captain of the French national team)
##
("entity"{tuple_delimiter}FIFA{tuple_delimiter}COMPETITION{tuple_delimiter}Fedration Internationale de Football Association, organiser of the World Cup)
##
("relationship"{tuple_delimiter}KYLIAN MBAPPE{tuple_delimiter}FRANCE{tuple_delimiter}Mbappe is the captain of the French national team{tuple_delimiter}9)
##
("relationship"{tuple_delimiter}FRANCE{tuple_delimiter}FIFA{tuple_delimiter}France is a member nation participating in FIFA's World Cup tournament{tuple_delimiter}8)
<|COMPLETE|>

---Real Data---
Entity types: {entity_types}
Text: {input_text}

Output:"""

print("Extraction prompt template defined.")
print(f"Template length: {len(EXTRACTION_PROMPT)} characters")

Extraction prompt template defined.
Template length: 1968 characters


In [5]:
def build_extraction_prompt(text: str) -> str:
    return EXTRACTION_PROMPT.format(
        tuple_delimiter=TUPLE_DELIMITER,
        record_delimiter=RECORD_DELIMITER,
        completion_delimiter=COMPLETION_DELIMITER,
        entity_types=ENTITY_TYPES,
        input_text=text,
    )

def call_llm(prompt: str) -> str:
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

print("Helper functions defined.")

Helper functions defined.


---
## Step 3 — Self-Reflection / Glean Pass  *(Appendix A.2)*

A single extraction pass often misses entities — especially those mentioned only briefly. The paper addresses this with a **"glean" pass** (Appendix A): after the initial extraction, the LLM is asked whether it missed anything. If it answers YES, we append a second extraction.

This is structurally similar to *chain-of-thought* self-critique: the model acts as both extractor and reviewer. The paper finds that one glean pass recovers a significant fraction of missed entities with minimal extra cost.

> **Implementation note:** We force a YES/NO answer first, then only call the model again if needed. One glean iteration is sufficient for this dataset.

In [6]:
GLEAN_CHECK_PROMPT = (
    "Many entities may have been missed in the last extraction.  "
    "Answer YES | NO: were important named entities of types {entity_types} missed from the text?"
)

GLEAN_EXTRACT_PROMPT = (
    "Add the missed entities and relationships using exactly the same tuple format:\n"
    '  ("entity"{tuple_delimiter}<name>{tuple_delimiter}<type>{tuple_delimiter}<description>)\n'
    '  ("relationship"{tuple_delimiter}<src>{tuple_delimiter}<tgt>{tuple_delimiter}<desc>{tuple_delimiter}<strength>)\n'
    "Separate with {record_delimiter} and end with {completion_delimiter}"
)


def extract_from_chunk(chunk_text: str, chunk_idx: int) -> str:
    """Run extraction + one glean pass on a single chunk. Returns raw LLM output."""
    prompt = build_extraction_prompt(chunk_text)
    result = call_llm(prompt)

    check = GLEAN_CHECK_PROMPT.format(entity_types=ENTITY_TYPES)
    yn = call_llm(f"{prompt}\n{result}\n\n{check}")

    if "YES" in yn.upper():
        glean_prompt = GLEAN_EXTRACT_PROMPT.format(
            tuple_delimiter=TUPLE_DELIMITER,
            record_delimiter=RECORD_DELIMITER,
            completion_delimiter=COMPLETION_DELIMITER,
        )
        extra = call_llm(f"{prompt}\n{result}\n\n{check}\n{yn}\n\n{glean_prompt}")
        result = (
            result.rstrip(COMPLETION_DELIMITER).rstrip()
            + "\n" + RECORD_DELIMITER + "\n"
            + extra
        )
        print(f"  chunk {chunk_idx:02d}: glean YES — appended extra tuples")
    else:
        print(f"  chunk {chunk_idx:02d}: glean NO")

    return result


print("Glean helpers defined.")

Glean helpers defined.


In [7]:
print(f"Running extraction over {len(chunks)} chunks — this takes ~3-5 min on qwen2.5:7b ...\n")

raw_outputs = []
for i, chunk in enumerate(chunks):
    out = extract_from_chunk(chunk, i)
    raw_outputs.append(out)

print(f"\nExtraction complete. Total raw output: {sum(len(o) for o in raw_outputs):,} characters")

Running extraction over 27 chunks — this takes ~3-5 min on qwen2.5:7b ...



  chunk 00: glean YES — appended extra tuples


  chunk 01: glean YES — appended extra tuples


  chunk 02: glean NO


  chunk 03: glean NO


...

  chunk 22: glean NO


  chunk 23: glean NO


  chunk 24: glean NO


  chunk 25: glean NO


  chunk 26: glean NO

Extraction complete. Total raw output: 108,411 characters


---
## Step 4 — Parse Tuples and Build the Graph  *(§ 3.1.3)*

Now we parse the raw LLM output into structured data and build a `networkx` graph.

**Merging strategy (§ 3.1.3):** When the same entity appears in multiple chunks, we merge by *exact case-insensitive name match*. Descriptions are concatenated so the merged node holds all context. For edges, if the same source–target pair appears multiple times, the edge weight increases — reflecting how strongly connected those entities are across the document.

This is deliberately simple: the paper notes that co-reference resolution could be added, but exact-name merging works well for a structured document like Wikipedia.

In [8]:
def parse_tuples(raw: str):
    """Extract entity and relationship tuples from raw LLM output."""
    entities      = []
    relationships = []

    raw = raw.split(COMPLETION_DELIMITER)[0]

    for record in raw.split(RECORD_DELIMITER):
        record = record.strip()
        if not record:
            continue

        # Unwrap outer parentheses if present
        m = re.match(r'^\((.+)\)\s*$', record, re.DOTALL)
        if m:
            record = m.group(1)

        parts = [p.strip().strip('"') for p in record.split(TUPLE_DELIMITER)]
        if not parts:
            continue

        kind = parts[0].lower().strip('"()')

        if kind == "entity" and len(parts) >= 4:
            entities.append({
                "name":        parts[1].upper().strip(),
                "type":        parts[2].upper().strip(),
                "description": parts[3].strip(),
            })
        elif kind == "relationship" and len(parts) >= 5:
            try:
                strength = int(re.search(r'\d+', parts[4]).group())
            except (AttributeError, ValueError):
                strength = 1
            relationships.append({
                "source":      parts[1].upper().strip(),
                "target":      parts[2].upper().strip(),
                "description": parts[3].strip(),
                "strength":    strength,
            })

    return entities, relationships


all_entities      = []
all_relationships = []

for raw in raw_outputs:
    ents, rels = parse_tuples(raw)
    all_entities.extend(ents)
    all_relationships.extend(rels)

print(f"Parsed {len(all_entities)} entity mentions, {len(all_relationships)} relationship mentions")

Parsed 329 entity mentions, 328 relationship mentions


In [9]:
# Merge entities by name (already uppercased)
entity_map = {}

for e in all_entities:
    key = e["name"]
    if key not in entity_map:
        entity_map[key] = {"type": e["type"], "descriptions": []}
    entity_map[key]["descriptions"].append(e["description"])

print(f"Unique entities after merging: {len(entity_map)}")

# Build networkx graph
G = nx.Graph()

for name, attrs in entity_map.items():
    G.add_node(
        name,
        type=attrs["type"],
        description=" | ".join(set(attrs["descriptions"])),
    )

edge_map = defaultdict(lambda: {"descriptions": [], "weight": 0})

for r in all_relationships:
    src, tgt = r["source"], r["target"]
    for n in (src, tgt):
        if n not in G.nodes:
            G.add_node(n, type="UNKNOWN", description="")
    key = tuple(sorted([src, tgt]))
    edge_map[key]["descriptions"].append(r["description"])
    edge_map[key]["weight"] += 1

for (src, tgt), attrs in edge_map.items():
    G.add_edge(
        src, tgt,
        description=" | ".join(set(attrs["descriptions"])),
        weight=attrs["weight"],
    )

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

top = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:10]
print("\nTop-10 by degree:")
for name, deg in top:
    print(f"  {name:<35} {deg:>3}  ({G.nodes[name].get('type', '?')})")

Unique entities after merging: 156
Graph: 179 nodes, 240 edges

Top-10 by degree:
  QATAR                                58  (NATION)
  FIFA                                 32  (COMPETITION)
  FRANCE                               18  (NATION)
  ARGENTINA                            16  (NATION)
  AUSTRALIA                             9  (NATION)
  BRAZIL                                8  (TEAM)
  LUSAIL STADIUM                        8  (STADIUM)
  IRAN                                  8  (NATION)
  ENGLAND                               7  (NATION)
  RUSSIA_NATIONAL_TEAM                  7  (TEAM)


In [10]:
graph_data = nx.node_link_data(G)
with open(GRAPH_PATH, "w", encoding="utf-8") as f:
    json.dump(graph_data, f, ensure_ascii=False, indent=2)

print(f"Graph saved to {GRAPH_PATH}")
print(f"File size: {GRAPH_PATH.stat().st_size / 1024:.1f} KB")

Graph saved to ../data/output/wc2022_trimmed_graph.json
File size: 107.6 KB


---
## Visualisation — Interactive Graph

We use `pyvis` to render the graph as an interactive HTML page. Rather than adding every node and edge by hand, we tag the `networkx` graph with a few display attributes (`color`, `size`, `title`) and hand it straight to `Network.from_nx()` — pyvis builds the whole visualisation in one call. Nodes are coloured by entity type so you can immediately see the structure: NATION and TEAM nodes tend to form clusters, while CONFEDERATION nodes act as hubs connecting them.

The output HTML embeds the vis.js library inline (`cdn_resources="in_line"`), so it opens as a standalone file.

In [11]:
from pyvis.network import Network
from IPython.display import IFrame

TYPE_COLORS = {
    "TEAM":          "#e63946",
    "NATION":        "#457b9d",
    "STADIUM":       "#2d6a4f",
    "CITY":          "#74c69d",
    "CONFEDERATION": "#f4a261",
    "COMPETITION":   "#9b2226",
    "PERSON":        "#8338ec",
    "EVENT":         "#ffb703",
    "UNKNOWN":       "#adb5bd",
}

# Only keep nodes that have at least one relationship — isolates are extraction
# noise and clutter the layout without contributing to GraphRAG communities.
connected = {n for n, d in G.degree() if d > 0}
G_vis = G.subgraph(connected).copy()
print(f"Visualising {G_vis.number_of_nodes()} connected nodes, "
      f"{G_vis.number_of_edges()} edges  "
      f"(dropped {G.number_of_nodes() - G_vis.number_of_nodes()} isolates)")

# Annotate the graph with pyvis display attributes, then let the library build the
# network in one call. pyvis reads these keys directly off each node/edge.
for node, attrs in G_vis.nodes(data=True):
    ntype  = attrs.get("type", "UNKNOWN")
    degree = G_vis.degree(node)
    attrs["label"] = node
    attrs["color"] = TYPE_COLORS.get(ntype, TYPE_COLORS["UNKNOWN"])
    attrs["size"]  = 12 + degree * 4
    attrs["title"] = (f"<b>{node}</b><br>Type: {ntype}<br>Degree: {degree}"
                      f"<br><br>{attrs.get('description', '')[:200]}")

for _, _, attrs in G_vis.edges(data=True):
    attrs["title"] = attrs.get("description", "")[:200]
    attrs["value"] = attrs.get("weight", 1) * 3

net = Network(
    height="700px", width="100%",
    bgcolor="#1a1a2e", font_color="white",
    notebook=True, cdn_resources="in_line",
)
net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=120)
net.from_nx(G_vis)                       # pyvis creates every node and edge for us

net.save_graph(str(HTML_PATH))
print(f"Visualisation saved to {HTML_PATH}")

IFrame(str(HTML_PATH), width="100%", height="720px")

Visualising 152 connected nodes, 240 edges  (dropped 27 isolates)
Visualisation saved to ../data/output/wc2022_trimmed_graph.html


---
## Summary

In this notebook we implemented §§ 3.1.1–3.1.3 of the GraphRAG paper:

| Paper section | What we did |
|---|---|
| § 3.1.1 | Chunked the PDF at 600 tok / 100 tok overlap |
| § 3.1.2 | Extracted entity and relationship tuples with a structured prompt |
| App. A.2 | Added a glean pass to recover missed entities |
| § 3.1.3 | Merged duplicates by name, built a weighted `networkx` graph |

The graph is saved at `data/output/wc2022_trimmed_graph.json` and `data/output/wc2022_trimmed_graph.html`.

**Next → Notebook 01:** We run the **Leiden** community detection algorithm on this graph to find clusters of closely related entities, mirroring § 3.1.4 of the paper.